# Agent vs Agent Matchup Heatmap

This notebook demonstrates how to visualize pairwise win rates between agents using a heatmap.

## What the Heatmap Shows

The heatmap displays the **win rate** of each agent (row) against each opponent (column):

- **Cell value**: Win rate of the row agent vs the column agent (0% to 100%)
- **Diagonal**: Grayed out (no self-play data)
- **Color scale**: Diverging colormap centered at 50% (green = row agent wins more, red = column agent wins more)

### Interpreting the Heatmap

- **Green cells**: The row agent beats the column agent more often than not
- **Red cells**: The column agent beats the row agent more often than not
- **Yellow cells**: Roughly even matchup (~50% win rate)

### Side-Balanced Aggregation

In games where the starting position matters (like Breakthrough), we can compute a **side-balanced** win rate that averages across both starting positions. This gives a fairer comparison when agents perform differently depending on which side they play.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.plot_matchup_heatmap import (
    compute_matchup_matrix_from_csv,
    plot_heatmap,
    save_matrix_csv,
)

## Load Results

We'll load match results from CSV files in the `results/` directory.

In [ ]:
# Find all CSV result files
results_dir = project_root / "results"
csv_files = sorted(results_dir.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files in {results_dir}")
if csv_files:
    print(f"Example files: {[f.name for f in csv_files[:3]]}")

## Compute the Matchup Matrix

The matchup matrix contains win rates for every pair of agents.

In [ ]:
# Compute matchup matrix from all CSV files
csv_paths = [str(f) for f in csv_files]
matrix = compute_matchup_matrix_from_csv(*csv_paths)

print(f"Game: {matrix.game or 'Multiple games'}")
print(f"Agents found ({len(matrix.agents)}): {', '.join(sorted(matrix.agents))}")

## Filter by Game (Optional)

If results contain multiple games, we can filter to a specific one.

In [ ]:
# Check what games are in the data
import csv

games_found = set()
for csv_file in csv_files:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            games_found.add(row.get("game", "unknown"))

print(f"Games in data: {sorted(games_found)}")

In [ ]:
# Filter to a specific game (e.g., breakthrough)
game_filter = "breakthrough" if "breakthrough" in games_found else None

if game_filter:
    matrix_filtered = compute_matchup_matrix_from_csv(*csv_paths, game_filter=game_filter)
    print(f"Filtered to {game_filter}: {len(matrix_filtered.agents)} agents")
else:
    matrix_filtered = matrix
    print("Using all games")

## Display the Win Rate Matrix

Let's look at the raw win rate data as a DataFrame.

In [ ]:
# Convert to DataFrame for easy viewing
sorted_agents = sorted(matrix_filtered.agents)
win_rates = matrix_filtered.to_matrix(side_balanced=False)

df = pd.DataFrame(win_rates, index=sorted_agents, columns=sorted_agents)
df.style.format("{:.0%}").background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)

## Plot the Heatmap

Now let's visualize the matchup data as a heatmap.

In [ ]:
# Create output directory
output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)

# Plot standard heatmap
output_path = output_dir / "matchup_heatmap_notebook.png"
plot_heatmap(
    matrix_filtered,
    output_path=output_path,
    side_balanced=False,
    annotate=True,
    show_samples=True,
)

# Display the image
from IPython.display import Image
Image(filename=output_path)

## Side-Balanced Heatmap

For games where starting position matters, the side-balanced heatmap shows fairer comparisons.

In [ ]:
# Plot side-balanced heatmap
output_path_balanced = output_dir / "matchup_heatmap_side_balanced_notebook.png"
plot_heatmap(
    matrix_filtered,
    output_path=output_path_balanced,
    side_balanced=True,
    annotate=True,
    show_samples=True,
)

Image(filename=output_path_balanced)

## Save the Matrix as CSV

Export the win rate matrix for further analysis.

In [ ]:
# Save to CSV
csv_path = output_dir / "matchup_matrix_notebook.csv"
save_matrix_csv(matrix_filtered, csv_path, side_balanced=False)

# Show the CSV content
print(csv_path.read_text())

## Sample Size Analysis

Check how many games were played between each agent pair.

In [ ]:
# Sample count matrix
sample_matrix = matrix_filtered.to_sample_matrix()
sample_df = pd.DataFrame(sample_matrix, index=sorted_agents, columns=sorted_agents)

print("Sample sizes (number of games between each agent pair):")
sample_df

## Summary Statistics

Compute overall performance metrics for each agent.

In [ ]:
# Calculate average win rate for each agent
summary_data = []

for agent in sorted_agents:
    # Get win rates against all opponents (excluding self)
    win_rates_vs_others = [
        matrix_filtered.get_win_rate(agent, opponent)
        for opponent in sorted_agents
        if opponent != agent
    ]
    
    # Filter out NaN values
    valid_rates = [r for r in win_rates_vs_others if not np.isnan(r)]
    
    if valid_rates:
        avg_win_rate = np.mean(valid_rates)
        min_win_rate = np.min(valid_rates)
        max_win_rate = np.max(valid_rates)
    else:
        avg_win_rate = min_win_rate = max_win_rate = np.nan
    
    # Total games
    total_games = sum(matrix_filtered.get_sample_count(agent, opp) for opp in sorted_agents) // 2
    
    summary_data.append({
        "Agent": agent,
        "Avg Win Rate": avg_win_rate,
        "Min Win Rate": min_win_rate,
        "Max Win Rate": max_win_rate,
        "Total Games": total_games,
    })

summary_df = pd.DataFrame(summary_data)
summary_df.style.format({
    "Avg Win Rate": "{:.1%}",
    "Min Win Rate": "{:.1%}",
    "Max Win Rate": "{:.1%}",
    "Total Games": "{:.0f}",
}).background_gradient(subset=["Avg Win Rate"], cmap="RdYlGn", vmin=0, vmax=1)

## Conclusions

The matchup heatmap provides a quick visual summary of agent performance:

1. **Strong agents** appear as green rows (high win rates against most opponents)
2. **Weak agents** appear as red rows (low win rates against most opponents)
3. **Non-transitive relationships** (e.g., A beats B, B beats C, C beats A) show up as patterns where no single agent dominates

### Next Steps

- Use the CLI script for batch processing: `python3 scripts/plot_matchup_heatmap.py results/*.csv`
- Compare with rating systems like Elo or Glicko-2 for overall rankings
- Use AlphaRank analysis for computing rankings in non-transitive games